In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME", "superstore_db")
DB_USER     = os.getenv("DB_USER", "superstore_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "superstore_pass")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [2]:
df = pd.read_csv("/home/jovyan/work/data/superstore_cleaned.csv")

print(df.shape)
print(df.dtypes)
df.head()

(9800, 22)
row_id             int64
order_id          object
order_date        object
ship_date         object
ship_mode         object
customer_id       object
customer_name     object
segment           object
country           object
city              object
state             object
postal_code        int64
region            object
product_id        object
category          object
sub-category      object
product_name      object
sales            float64
year             float64
month            float64
quarter          float64
delivery_days    float64
dtype: object


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,region,product_id,category,sub-category,product_name,sales,year,month,quarter,delivery_days
0,1,CA-2017-152156,2017-08-11,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2017.0,8.0,3.0,92.0
1,2,CA-2017-152156,2017-08-11,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,2017.0,8.0,3.0,92.0
2,3,CA-2017-138688,2017-12-06,NaN,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2017.0,12.0,4.0,NaN
3,4,US-2016-108966,2016-11-10,NaN,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,2016.0,11.0,4.0,NaN
4,5,US-2016-108966,2016-11-10,NaN,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2016.0,11.0,4.0,NaN


In [3]:

regions = df[['region', 'country', 'state', 'city', 'postal_code']].drop_duplicates().reset_index(drop=True)
regions.insert(0, 'region_id', regions.index + 1)


customers = df[['customer_id', 'customer_name', 'segment']].drop_duplicates().reset_index(drop=True)


categories = df[['category', 'sub-category']].drop_duplicates().reset_index(drop=True)
categories.insert(0, 'category_id', categories.index + 1)
categories.rename(columns={'sub-category': 'sub_category'}, inplace=True)

products = df[['product_id', 'product_name', 'category', 'sub-category']].drop_duplicates('product_id').reset_index(drop=True)
products.rename(columns={'sub-category': 'sub_category'}, inplace=True)
products = products.merge(categories, on=['category', 'sub_category'], how='left')
products = products[['product_id', 'product_name', 'category_id']]

print(f"Regions:    {len(regions)}")
print(f"Customers:  {len(customers)}")
print(f"Categories: {len(categories)}")
print(f"Products:   {len(products)}")

Regions:    628
Customers:  793
Categories: 17
Products:   1861


In [4]:

df2 = df.merge(
    regions, on=['region', 'country', 'state', 'city', 'postal_code'], how='left'
)


orders = df2[[
    'order_id', 'order_date', 'ship_date', 'ship_mode',
    'customer_id', 'region_id', 'year', 'month', 'quarter'
]].drop_duplicates('order_id').reset_index(drop=True)


order_items = df2[[
    'row_id', 'order_id', 'product_id', 'sales', 'delivery_days'
]].copy()
order_items.rename(columns={'row_id': 'item_id'}, inplace=True)

print(f"Orders:      {len(orders)}")
print(f"Order Items: {len(order_items)}")

Orders:      4922
Order Items: 9800


In [5]:
create_tables_sql = """
-- REGIONS
CREATE TABLE IF NOT EXISTS regions (
    region_id   SERIAL PRIMARY KEY,
    region      VARCHAR(50),
    country     VARCHAR(100),
    state       VARCHAR(100),
    city        VARCHAR(100),
    postal_code VARCHAR(20)
);

-- CUSTOMERS
CREATE TABLE IF NOT EXISTS customers (
    customer_id   VARCHAR(20) PRIMARY KEY,
    customer_name VARCHAR(100),
    segment       VARCHAR(50)
);

-- CATEGORIES
CREATE TABLE IF NOT EXISTS categories (
    category_id  SERIAL PRIMARY KEY,
    category     VARCHAR(100),
    sub_category VARCHAR(100)
);

-- PRODUCTS
CREATE TABLE IF NOT EXISTS products (
    product_id   VARCHAR(50) PRIMARY KEY,
    product_name VARCHAR(255),
    category_id  INT REFERENCES categories(category_id)
);

-- ORDERS
CREATE TABLE IF NOT EXISTS orders (
    order_id    VARCHAR(30) PRIMARY KEY,
    order_date  DATE,
    ship_date   DATE,
    ship_mode   VARCHAR(50),
    customer_id VARCHAR(20) REFERENCES customers(customer_id),
    region_id   INT         REFERENCES regions(region_id),
    year        INT,
    month       INT,
    quarter     INT
);

-- ORDER ITEMS (Fact Table)
CREATE TABLE IF NOT EXISTS order_items (
    item_id       INT PRIMARY KEY,
    order_id      VARCHAR(30) REFERENCES orders(order_id),
    product_id    VARCHAR(50) REFERENCES products(product_id),
    sales         NUMERIC(10,2),
    delivery_days INT
);
"""

with engine.connect() as conn:
    conn.execute(text(create_tables_sql))
    conn.commit()
    print("All tables created!")

All tables created!


In [6]:
regions.to_sql('regions',     engine, if_exists='append', index=False)
print("regions loaded")

customers.to_sql('customers', engine, if_exists='append', index=False)
print("customers loaded")

categories.to_sql('categories', engine, if_exists='append', index=False)
print("categories loaded")

products.to_sql('products',   engine, if_exists='append', index=False)
print("products loaded")

orders.to_sql('orders',       engine, if_exists='append', index=False)
print("orders loaded")

order_items.to_sql('order_items', engine, if_exists='append', index=False)
print("order_items loaded")

print("\n All data loaded successfully!")

regions loaded
customers loaded
categories loaded
products loaded
orders loaded
order_items loaded

 All data loaded successfully!


In [7]:
checks = {
    "regions":     "SELECT COUNT(*) FROM regions",
    "customers":   "SELECT COUNT(*) FROM customers",
    "categories":  "SELECT COUNT(*) FROM categories",
    "products":    "SELECT COUNT(*) FROM products",
    "orders":      "SELECT COUNT(*) FROM orders",
    "order_items": "SELECT COUNT(*) FROM order_items",
}

with engine.connect() as conn:
    for table, query in checks.items():
        result = conn.execute(text(query)).scalar()
        print(f"  {table:<15} → {result:>6} rows")

  regions         →    628 rows
  customers       →    793 rows
  categories      →     17 rows
  products        →   1861 rows
  orders          →   4922 rows
  order_items     →   9800 rows


In [8]:
queries = {
    "Total Sales by Category": """
        SELECT c.category, ROUND(SUM(oi.sales)::numeric, 2) AS total_sales
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        GROUP BY c.category
        ORDER BY total_sales DESC;
    """,

    "Sales by Region": """
        SELECT r.region, ROUND(SUM(oi.sales)::numeric, 2) AS total_sales
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN regions r ON o.region_id = r.region_id
        GROUP BY r.region
        ORDER BY total_sales DESC;
    """,

    "Top 10 Customers by Revenue": """
        SELECT cu.customer_name, ROUND(SUM(oi.sales)::numeric, 2) AS revenue
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN customers cu ON o.customer_id = cu.customer_id
        GROUP BY cu.customer_name
        ORDER BY revenue DESC
        LIMIT 10;
    """,

    "Number of Orders per Month": """
        SELECT year, month, COUNT(DISTINCT order_id) AS nb_orders
        FROM orders
        GROUP BY year, month
        ORDER BY year, month;
    """,

    "Average Delivery Days by Ship Mode": """
        SELECT o.ship_mode, ROUND(AVG(oi.delivery_days)::numeric, 1) AS avg_days
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        GROUP BY o.ship_mode
        ORDER BY avg_days;
    """,
}

with engine.connect() as conn:
    for title, sql in queries.items():
        print(f"\n{'='*50}")
        print(f"{title}")
        print('='*50)
        result = pd.read_sql(text(sql), conn)
        print(result.to_string(index=False))


Total Sales by Category
       category  total_sales
     Technology    827455.94
      Furniture    728658.75
Office Supplies    705422.28

Sales by Region
 region  total_sales
   West    710219.77
   East    669518.85
Central    492646.90
  South    389151.45

Top 10 Customers by Revenue
     customer_name  revenue
       Sean Miller 25043.07
      Tamara Chand 19052.22
      Raymond Buch 15117.35
      Tom Ashbrook 14595.62
     Adrian Barton 14473.57
      Ken Lonsdale 14175.23
      Sanjit Chand 14142.34
      Hunter Lopez 12873.30
      Sanjit Engle 12209.44
Christopher Conant 12129.08

Number of Orders per Month
  year  month  nb_orders
2015.0    1.0         35
2015.0    2.0         32
2015.0    3.0         31
2015.0    4.0         29
2015.0    5.0         35
2015.0    6.0         31
2015.0    7.0         31
2015.0    8.0         35
2015.0    9.0         32
2015.0   10.0         19
2015.0   11.0         35
2015.0   12.0         30
2016.0    1.0         32
2016.0    2.0         

In [9]:
views_sql = """
CREATE OR REPLACE VIEW v_sales_by_category AS
    SELECT c.category, c.sub_category,
           ROUND(SUM(oi.sales)::numeric, 2) AS total_sales,
           COUNT(oi.item_id) AS nb_items
    FROM order_items oi
    JOIN products p   ON oi.product_id = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY c.category, c.sub_category;

CREATE OR REPLACE VIEW v_sales_by_region AS
    SELECT r.region, r.state,
           ROUND(SUM(oi.sales)::numeric, 2) AS total_sales,
           COUNT(DISTINCT o.order_id) AS nb_orders
    FROM order_items oi
    JOIN orders o  ON oi.order_id = o.order_id
    JOIN regions r ON o.region_id = r.region_id
    GROUP BY r.region, r.state;

CREATE OR REPLACE VIEW v_top_customers AS
    SELECT cu.customer_id, cu.customer_name, cu.segment,
           ROUND(SUM(oi.sales)::numeric, 2) AS total_revenue,
           COUNT(DISTINCT o.order_id) AS nb_orders
    FROM order_items oi
    JOIN orders o    ON oi.order_id = o.order_id
    JOIN customers cu ON o.customer_id = cu.customer_id
    GROUP BY cu.customer_id, cu.customer_name, cu.segment
    ORDER BY total_revenue DESC;
"""

with engine.connect() as conn:
    conn.execute(text(views_sql))
    conn.commit()
    print("Views created: v_sales_by_category, v_sales_by_region, v_top_customers")

Views created: v_sales_by_category, v_sales_by_region, v_top_customers
